<a href="https://colab.research.google.com/github/Pale24/GasBioLab/blob/main/RDF3_GEO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab RF3 para proyecto de diseño de Binders del Gaston Biotech Lab S.A

## Información del [manual](https://rosettacommons.github.io/foundry/index.html)

## Inference Calculation Basics
En RFdiffusion3 (RFD3), se utilizan archivos YAML o JSON para especificar la configuración de los cálculos de inferencia, y las opciones de configuración se utilizan para proporcionar otra información sobre el cálculo, como la ubicación y el nombre del archivo de punto de control que se desea utilizar.

### Inference Settings

The inference 'settings' are how you constrain your inference calculation, such as specifying portions of the output you wish to have designed (`contig`) and specifying any symmetries that exist in your system (`symmetry`). These settings are stored in either a YAML or JSON file to be interpreted by RFdiffusion3. Runnable examples of json and yaml files can be found in foundry/models/rfd3/docs/examples.

Using this type of input specification allows you to define different types of inference calculations all in the same file, and either run all of the calculation types defined in the file or specify the specific calculation you want to run via the command line.

#### Job configurations
Once you have all of the settings you want to use to constrain your inference run in a JSON or YAML file, you can run the job using a command starting with rfd3 design and then including different 'configuration options'. You must include the path to the YAML/JSON file that defines your inference run(s) and the output directory:

```
rfd3 design inputs=/path/to/your/yaml/or/json/file out_dir=/path/to/your/output/directory ckpt_path=/path/to/an/rfd3_checkpoint_file.pt
```
Several options are available to you as well to control the number of designs, whether to save the trajectory files, etc. These options can be found in "foundry/models/rfd3/configs/inference_engine/base.yaml" and "foundry/models/rfd3/configs/inference_engine/rfdiffusion3.yaml"

#### Output Files
At the end of your inference calculation, you will be left with several output files in the directory you specified. At minimum (if you did not change any settings to include more outputs) you will be left with a JSON and a compressed CIF file (.cif.gz) for each design. The names of the files will be as follows:

```
<name of the json or yaml file>_<settings group name>_<batch_number>_model_n.<suffix>
```

Where n is the design number, the numbering for the designs will start at 0.

For an example, if I called the my JSON file rfd3_example.json, only ran one batch, and had a group of settings in it labeled example_1 I would get files with names like:

```
rfd3_example_example_1_0_model_0.cif.gz
rfd3_example_example_1_0_model_0.json
rfd3_example_example_1_0_model_1.cif.gz
rfd3_example_example_1_0_model_1.json
...

```

### Input Specification & Command-line arguments

RFdiffusion3 accepts inputs in two forms:

- Constrains to be applied to the inference run are given in JSON or YAML files
- Details about the job (number of designs, output directory, etc.) are given as command line arguments

This document outlines the various input settings and configurations you can use with RFdiffusion3.

JSON inputs take the following top-level structure;

```json
{
    "spec-1": {  // First design configuration
      "input": "<path/to/pdb>",
      "contig": "50-80,/0,A1-100",  // Diffuses length 50-80 monomer in chain A & selects indices A1 -> A100 in input pdb to have fixed coordinates and sequences  
      "select_unfixed_sequence": "A20-35", // Converts selected indices in input to have unfixed sequence (inputs become atom14).
      "ligand": "HAX,OAA",  // Selects ligands HAX and OAA based on res name in the input
    },
    "spec-2": {
      // ... args for the second (independent) configuration for design.
    }
}
```

You can then run inference at the command line with:
```
rfd3 design out_dir=<path/to/outdir> inputs=<path/to/inputs>
```

**Required CLI arguments:**

- `out_dir` — The directory that output files from the inference run will be stored in. If the directory does not exist it will be created. This does not change how the output files are named.
- `inputs` — The path and file name of the JSON or YAML file where you have defined your inference constraints.

**Other Useful CLI arguments:**

- `n_batches` — number of batches to generate per input key (default: 1).
- `diffusion_batch_size` — number of diffusion samples (designs) per batch (default: 8). If `n_batches=1` and `diffusion_batch_size=8` then 8 designs will be generated from the inference run.
- `specification`— JSON overrides for the per-example InputSpecification (default: {}). For example, you can run rfd3 design inputs=null specification.length=200 for a quick debug of creating a 200-length protein.
- `inference_sampler.num_timesteps` — diffusion timesteps for sampling (default: 200).
- `inference_sampler.step_scale` — scales diffusion step size; higher → less diverse, more designable (default: 1.5).
- `low_memory_mode` — memory-efficient tokenization mode; set True if GPU RAM is tight (default:`False).
- `ckpt_path` — String containing the path and file name of the checkpoint path you want to use (default: rfd3)
- `skip_existing` — Skip designing any systems whose output files already exist in the specified out_dir (default: True).
- `global_prefix` — This setting allows you to change the beginning of the name of the output files from the name of the input JSON or YAML file to your own string (default: null).
- `dump_trajectories` — If True, the trajectory files are also saved to the specified output directory (default: False).
- `prevalidate_inputs` — Check that your inputs (JSON or YAML file) are valid before running inference (default: False).
- `low_memory_mode` - Set to True (default: False) for memory efficient tokenization mode.





In [ ]:
# Instalar Foundry
!pip install rc-foundry[all]

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.6 MB/s eta 0:00:00


In [ ]:
try:
  import py3Dmol
except:
  !pip install py3Dmol
  import py3Dmol

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
HOME_PATH = '/content/drive/MyDrive/bioinformatica'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Instalamos el modelo de RF3 para foundry
import os

CHECKPOINT_PATH = f'{HOME_PATH}/foundry/checkpoints'
CHECKPOINT_RFD3 = f'{CHECKPOINT_PATH}/rfd3_latest.ckpt'

if not os.path.exists(CHECKPOINT_RFD3):
    print("Descargando checkpoint de RF3...")
    os.makedirs(f'{CHECKPOINT_PATH}', exist_ok=True)
    os.system(f"foundry install rfd3 ligandmpnn --checkpoint-dir {CHECKPOINT_PATH}")

In [ ]:
# Shared utilities for visualization (from AtomWorks)
from atomworks.io.utils.visualize import view

  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository


In [ ]:
from lightning.fabric import seed_everything
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine
from rfd3.inference.input_parsing import DesignInputSpecification

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
INFO:foundry:cuEquivariance is available and will be used.
DEBUG:transforms:Debug mode is on


In [ ]:
# ============================================================
# 1. Establecer el directorio de trabajo
# ============================================================
FOUNDRY_PATH = f'{HOME_PATH}/foundry'
REPO_PATH = f'{FOUNDRY_PATH}/gas04'
INPUT_PATH = f'{REPO_PATH}/inputs'
INPUT_FILE = 'ligando6UNK.pdb'

print(f'{FOUNDRY_PATH}\n{REPO_PATH}\n{INPUT_PATH}')
print(f"✅ Usando estructura: {INPUT_FILE[:-4]}")

/content/drive/MyDrive/bioinformatica/foundry
/content/drive/MyDrive/bioinformatica/foundry/gas04
/content/drive/MyDrive/bioinformatica/foundry/gas04/inputs
✅ Usando estructura: ligando6UNK


In [ ]:
# ============================================================
# Verificar que el directorio de trabajo existe
# ============================================================
if not os.path.exists(REPO_PATH):
  print(f"⚠️ Directorio {REPO_PATH} no encontrado en la ruta esperada. Creando...")
  os.makedirs(f'{REPO_PATH}', exist_ok=True)

In [ ]:
# ============================================================
# Verificar que el input existe
# ============================================================
if not os.path.exists(INPUT_PATH) and os.path.exists(f'{HOME_PATH}/inputs'):
    print(f"⚠️ Directorio {INPUT_PATH} no encontrado en la ruta {REPO_PATH}. Creando...")
    os.makedirs(f'{INPUT_PATH}', exist_ok=True)
    print(f"⚠️ Copiando {INPUT_FILE} a {INPUT_PATH}")
    os.system(f"cp {HOME_PATH}/inputs/{INPUT_FILE} {INPUT_PATH}/{INPUT_FILE}")

In [ ]:
# Visualizar el input
with open(f"{INPUT_PATH}/{INPUT_FILE}", "r") as f:
    pdb_data = f.read()

viewer = py3Dmol.view(width=800, height=600)
viewer.addModel(pdb_data, "pdb")
viewer.setBackgroundColor('black')
#viewer.setStyle({'cartoon': {'color': 'spectrum'}})
viewer.setStyle({'chain':'B'}, {'stick': {'colorscheme':'yellowCarbon'}})
viewer.addPropertyLabels(
    "atom",  # Esta palabra clave le indica a py3Dmol que use el nombre del átomo
    {},
    {
        'fontSize': 10,
        'fontColor': 'white',
        'backgroundColor': 'black',
        'backgroundOpacity': 0.6,
        'alignment': 'center'
    },
    {}  # Selección vacía aplica a TODO el modelo
)
viewer.zoomTo()
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
#Seleccionar las carpetas donde se guardan las salidas
RESULTS_PATH = f"{REPO_PATH}/outputs"
RESULTS_DIR = "UNK-lig"

In [ ]:
# Set seed for reproducibility
seed_everything(0)

# Configure RFD3 inference
config = RFD3InferenceConfig(
    ckpt_path=f'{CHECKPOINT_PATH}/rfd3_latest.ckpt',
    specification={
        'extra': {},  # We are not using any extra specifications here.
    },
    diffusion_batch_size=5,  # Genera 3 estructuras por batch
)

spec = DesignInputSpecification(
    input=f'{INPUT_PATH}/{INPUT_FILE}',
    #length=150,
    #allow_ligand_on_existing_chain=True,
    ligand='UNK',
    dialect = 2,
    #infer_ori_strategy = "ligand",
    contig = "100-200",
    select_hotspots = {
        "UNK": "C1,C2,C3,C4,C5,C6,O1,O2,O3,O4,O5,O6,N1,CA1,C7,CB1,CG1,CG2,N2,CA2,CB2,CG3,CD2,NE2,CE1,ND1,C8,O7,N3",
    },
    select_buried = {
        "UNK": "C1,C2,C3,C4,C5,C6,O1,O2,O3,O4,O5,O6,N1,CA1,C7,CB1,CG1,CG2,N2,CA2,CB2,CG3,CD2,NE2,CE1,ND1,C8,O7,N3",
    },
    select_exposed = {
        "UNK": "N5,CA5,C11,CB5,CG5,CD,N6,CA6,C12,CB6,CG7,CD4,OE2,OE1,O10,O11,OXT",
    },
    is_non_loopy = True
    #unindex='A92-96,B2,E5',
    #select_fixed_atoms={
    #    'B2': 'ND1,CG,NE2,CD2,CB,CE1',
    #    'E5': "OH,CZ,CE1,CE2,CD2,CD1,CG,CB"
    #}
)

INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0


In [ ]:
# Initialize engine and run generation
model = RFD3InferenceEngine(**config)
outputs = model.run(
    inputs=spec,                                 # None for unconditional generation
    out_dir=f'{RESULTS_PATH}/{RESULTS_DIR}',     # None to return in memory (no file output)
    n_batches=5,                                 # Generate 5 batch
)

INFO:rfd3.engine:[rank: 0] Outputs will be written to /content/drive/MyDrive/bioinformatica/foundry/gas04/outputs/UNK-lig.
INFO:rfd3.engine:[rank: 0] Found 0 existing example IDs in the output directory (0 total).
INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:rfd3.engine:[rank: 0] Finished inference batch in 116.33 seconds.
INFO:rfd3.engine:[rank: 0] Outputs for backbone_0_4_model_0 written to /content/drive/MyDrive/bioinformatica/foundry/gas04/outputs/UNK-lig/backbone_0_4_model_0.
INFO:rfd3.engine:[rank: 0] Outputs for backbone_0_4_model_1 written to /content/drive/MyDrive/bioinformatica/foundry/gas04/outputs/UNK-lig/backbone_0_4_model_1.
INFO:rfd3.engine:[rank: 0] Outputs for backbone_0_4_model_2 written to /content/drive/MyDrive/bioinformatica/foundry/gas04/outputs/UNK-lig/backbone_0_4_model_2.
INFO:rfd3.engine:[rank: 0] Outputs for backbone_0_4_model_3 written to /content/drive/MyD

In [ ]:
import os
import json
import glob
import pandas as pd
from IPython.display import display, HTML
from google.colab import files

# =======================================================
# CONFIGURACIÓN (ajusta según tu caso)
# =======================================================
#RESULTS = f"{RESULTS_PATH}/{RESULTS_DIR}"
TOP_N = 10                        # Número de mejores diseños a descargar

# Criterios de selección (ordena por estas métricas)
# - 'max_ca_deviation' menor es mejor (estructura más cercana al diseño)
# - 'ligand_min_distance' mayor es mejor (evita choques con el ligando)
# - 'loop_fraction' menor es mejor (estructura más compacta)
# - 'radius_of_gyration' intermedio (depende del tamaño)
SORT_BY = 'max_ca_deviation'      # Métrica principal para ordenar
#SORT_BY = 'loop_fraction'        # Métrica principal para ordenar
ASCENDING = True                  # True si menor es mejor, False si mayor es mejor

# =======================================================
# 1. LEER TODOS LOS JSON Y EXTRAER MÉTRICAS
# =======================================================
json_files = glob.glob(os.path.join(f"{RESULTS_PATH}/{RESULTS_DIR}", "*.json"))
if not json_files:
    raise FileNotFoundError(f"No se encontraron archivos JSON en {RESULTS_DIR}")

data_rows = []
for json_path in sorted(json_files):
    with open(json_path, 'r') as f:
        data = json.load(f)

    # Extraer métricas del campo "metrics"
    metrics = data.get("metrics", {})
    spec = data.get("specification", {})
    extra = spec.get("extra", {})

    row = {
        "archivo": os.path.basename(json_path),
        "max_ca_deviation": metrics.get("max_ca_deviation", None),
        "n_chainbreaks": metrics.get("n_chainbreaks", None),
        "ligand_min_distance": metrics.get("n_clashing.ligand_min_distance", None),
        "ligand_clashes": metrics.get("n_clashing.ligand_clashes", None),
        "loop_fraction": metrics.get("loop_fraction", None),
        "helix_fraction": metrics.get("helix_fraction", None),
        "sheet_fraction": metrics.get("sheet_fraction", None),
        "num_ss_elements": metrics.get("num_ss_elements", None),
        "radius_of_gyration": metrics.get("radius_of_gyration", None),
        "num_residues": metrics.get("num_residues", None),
        "sample_id": extra.get("example_id", "N/A"),
    }
    # Si hay más métricas, puedes añadirlas aquí
    data_rows.append(row)

# Crear DataFrame
df = pd.DataFrame(data_rows)

# =======================================================
# 2. ORDENAR POR EL CRITERIO SELECCIONADO
# =======================================================
# Filtrar filas con valores nulos en la métrica principal
df_filtered = df[df[SORT_BY].notna()].copy()

if df_filtered.empty:
    print(f"⚠️ No hay diseños con la métrica '{SORT_BY}' disponible. Usando todas.")
    df_filtered = df

# Ordenar
df_sorted = df_filtered.sort_values(by=SORT_BY, ascending=ASCENDING).reset_index(drop=True)

# =======================================================
# 3. MOSTRAR TABLA COMPLETA
# =======================================================
print("📊 Tabla de métricas de los diseños (ordenada por criterio):")
# Mostrar solo las columnas más relevantes
display_cols = ["archivo", "max_ca_deviation", "ligand_min_distance",
                "loop_fraction", "num_residues", "sample_id"]
display(df_sorted[display_cols])

# Guardar tabla completa como CSV
df_sorted.to_csv("metricas_completas.csv", index=False)
print("✅ Tabla completa guardada como 'metricas_completas.csv'")

# =======================================================
# 4. SELECCIONAR LOS TOP N
# =======================================================
top_df = df_sorted.head(TOP_N).copy()
print(f"\n🏆 Top {TOP_N} candidatos seleccionados:")
display(top_df[display_cols])
mejores = top_df['archivo'].astype(str).str[:-5].tolist()


# Seleccionar el nombre del mejor modelo
mejor = top_df.iloc[0]['archivo'][:-5]
print(f"\n🏆 Mejor modelo seleccionado: {mejor}")



📊 Tabla de métricas de los diseños (ordenada por criterio):


,archivo,max_ca_deviation,ligand_min_distance,loop_fraction,num_residues,sample_id
0,backbone_0_1_model_2.json,0.037190,None,0.135802,222,backbone_0_1
1,backbone_0_0_model_0.json,0.041301,None,0.135338,193,backbone_0_0
2,backbone_0_0_model_1.json,0.046669,None,0.112782,193,backbone_0_0
3,backbone_0_4_model_2.json,0.049074,None,0.143791,213,backbone_0_4
4,backbone_0_3_model_4.json,0.053115,None,0.130000,260,backbone_0_3
5,backbone_0_2_model_2.json,0.054667,None,0.093168,221,backbone_0_2
6,backbone_0_2_model_4.json,0.055940,None,0.130435,221,backbone_0_2
7,backbone_0_1_model_1.json,0.058531,None,0.086420,222,backbone_0_1
8,backbone_0_3_model_1.json,0.058725,None,0.080000,260,backbone_0_3
9,backbone_0_0_model_2.json,0.059321,None,0.090226,193,backbone_0_0


✅ Tabla completa guardada como 'metricas_completas.csv'

🏆 Top 10 candidatos seleccionados:


,archivo,max_ca_deviation,ligand_min_distance,loop_fraction,num_residues,sample_id
0,backbone_0_1_model_2.json,0.037190,None,0.135802,222,backbone_0_1
1,backbone_0_0_model_0.json,0.041301,None,0.135338,193,backbone_0_0
2,backbone_0_0_model_1.json,0.046669,None,0.112782,193,backbone_0_0
3,backbone_0_4_model_2.json,0.049074,None,0.143791,213,backbone_0_4
4,backbone_0_3_model_4.json,0.053115,None,0.130000,260,backbone_0_3
5,backbone_0_2_model_2.json,0.054667,None,0.093168,221,backbone_0_2
6,backbone_0_2_model_4.json,0.055940,None,0.130435,221,backbone_0_2
7,backbone_0_1_model_1.json,0.058531,None,0.086420,222,backbone_0_1
8,backbone_0_3_model_1.json,0.058725,None,0.080000,260,backbone_0_3
9,backbone_0_0_model_2.json,0.059321,None,0.090226,193,backbone_0_0



🏆 Mejor modelo seleccionado: backbone_0_1_model_2


In [ ]:
i = 0
for file in mejores:
  print(i, file)
  i += 1


0 backbone_0_1_model_2
1 backbone_0_0_model_0
2 backbone_0_0_model_1
3 backbone_0_4_model_2
4 backbone_0_3_model_4
5 backbone_0_2_model_2
6 backbone_0_2_model_4
7 backbone_0_1_model_1
8 backbone_0_3_model_1
9 backbone_0_0_model_2


In [ ]:
#from atomworks.io.parser import parse
# Shared utilities for visualization (from AtomWorks)
#from atomworks.io.utils.visualize import view


#result = parse(f"{RESULTS_DIR}/{mejores[2]}.cif.gz")
#result.keys()
#result
#atom_array = result['assemblies']['1']
#print(atom_array)
#view(atom_array)


In [ ]:
import gzip

# Load CIF data from a local file
with gzip.open(f"{RESULTS_PATH}/{RESULTS_DIR}/{mejores[4]}.cif.gz", "rt") as f:
    cif_data = f.read()

# Initialize viewer
view = py3Dmol.view(width=800, height=600)

# Add CIF model
view.addModel(cif_data, "cif")

# Set style (e.g., stick representation)
view.setBackgroundColor('black')
view.setStyle({'chain':'A'}, {'cartoon': {'color': 'spectrum'}})
view.setStyle({'chain':'B'}, {'stick': {'colorscheme':'yellowCarbon'}})
view.addSurface(py3Dmol.VDW, {'opacity':0.8, 'color':'grey'}, \
  {'not':{'chain':'B'}})
#view.setStyle({"stick": {}})

# Zoom and render
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# Despues de observar las estructuras, seleccionamos el backbone que se ajuste a lo que estamos buscando:
# - Que interactue principalmente con el azucar
# - Que el extremo Carboxilo terminal quede expuesto al solvente ya que se supone que continua el resto de la proteina
seleccionado = mejores[4]
print(seleccionado)

backbone_0_3_model_4


# LIGANDMPNN
LigandMPNN uses JSON and JSONL files to pass advanced constraints, multi-structure batches, and residue-specific parameters to the [LigandMPNN GitHub Repository](https://github.com/dauparas/LigandMPNN) script (run.py). [1, 2]

## Key JSON/JSONL Options Explained

## 1. --pdb_path_multi (Multiple PDB Batch Input)

* What it does: Accepts a JSON file mapping multiple PDB file paths so you can run batch jobs without reloading model weights repeatedly.
* Format example:
```
`
  "./inputs/1BC8.pdb": "",
  "./inputs/4GYT.pdb": ""
}
```
[1]

## 2. --fixed_residues_multi or --fixed_positions_jsonl (Fixed Residues)

* What it does: Locks specific amino acid positions so they are not redesigned during sequence generation.
* Format for multi-PDB (--fixed_residues_multi): Maps each PDB path to a string of space-separated chain and residue IDs.

```
{
  "./inputs/1BC8.pdb": "C1 C2 C3 C4 C5 C10 C22",
  "./inputs/4GYT.pdb": "A7 A8 A9 A10 A11 A12 A13 B38"
}
```

* Format for single PDB (--fixed_positions_jsonl): Uses dictionary arrays per chain ID:

{"A": [1, 2, 3, 10, 11], "B": [5, 6]}

[2, 3, 4]

## 3. --bias_AA_jsonl (Global Amino Acid Biases)

* What it does: Adjusts the global probability of specific amino acids across the entire designed sequence. Positive numbers increase likelihood; negative numbers decrease it.
* Format example:

```
{"`": -1.1, "F": 0.7}
```
[5]

## 4. --bias_by_res_jsonl (Per-Position Biases)

* What it does: Applies specific amino acid biases to individual residue positions rather than globally.
* Format example:
```
{
  "A": {
    "10": {"A": 2.0, "G": -2.0},
    "11": {"R": 1.5}
  }
}
```

## 5. --omit_AA_jsonl (Per-Position Omissions)

* What it does: Restricts specific residues from sampling designated amino acids on a position-by-position basis.
* Format example:

```
{
  "A": {
    "15": "C",
    "20": "MFW"
  }
}
```
[5]

- [1] [https://github.com](https://github.com/dauparas/LigandMPNN)
- [2] [https://context7.com](https://context7.com/dauparas/ligandmpnn)
- [3] [https://docs.nvidia.com](https://docs.nvidia.com/nim/bionemo/proteinmpnn/latest/endpoints.html)
- [4] [https://proteinbase.com](https://proteinbase.com/protein-design-skills/proteinmpnn)
- [5] [https://huggingface.co](https://huggingface.co/spaces/simonduerr/ProteinMPNN/blob/main/ProteinMPNN/README.md)


When configured via the primary entry points of the [LigandMPNN GitHub repository](https://github.com/dauparas/LigandMPNN), the JSON and command-line flags contain several highly specific options that dictate how the underlying graph neural network (GNN), autoregressive decoder, and sampling algorithms operate.
The algorithmic parameters can be grouped into model architecture switches, decoding controls, and ligand-interaction behavior.

------------------------------
## 1. Decoding & Sampling Algorithm Options
These options alter the autoregressive sampling process used to decode sequences from the learned structural representations.

| JSON / CLI Parameter | Default Value | Description |
|---|---|---|
| --sampling_temp | 0.1 | Controls the softmax temperature for sampling amino acids. Lower values (e.g., 0.05–0.1) generate high-confidence, conservative sequences close to the argmax. Higher values (e.g., 0.5–1.0) increase sequence diversity but may reduce structural compatibility. |
| --seed | 0 | Sets the random seed for the sampling algorithm, ensuring reproducibility when generating multiple sequence variations for the same input backbone. |
| --batch_size | 1 | Controls how many sequences the sampling algorithm decodes in a single forward pass parallel batch. Higher batch sizes speed up large-scale structural generation. |

------------------------------
## 2. Model Backbone & Weights Options
These options switch between different algorithmic checkpoints trained under distinct physics and structural assumptions.

| JSON / CLI Parameter | Checkpoint Variations | Algorithmic Impact |
|---|---|---|
| --model_type | protein_mpnn, ligand_mpnn | Switches the underlying network algorithm. protein_mpnn relies entirely on backbone coordinates ($\text{N, C}_{\alpha}\text{, C, O}$). ligand_mpnn activates specialized message-passing layers capable of digesting non-protein atom coordinates, explicit chemical bonds, and element types. |
| --checkpoint_path | Path to .pt files | Points to the trained weights. Choosing a checkpoint trained with soluble proteins vs. membrane proteins changes the underlying structural propensity vectors predicted by the network. |

------------------------------
## 3. Ligand & Environment Graph Processing
When running LigandMPNN, these flags directly control how the graph construction algorithm treats non-protein atoms, ions, and water molecules.


* --use_soluble_model (Boolean): Forces the algorithm to interpret the structural environment using rules optimized for soluble, globular protein environments.
* --keep_water (Boolean): If set to true, the graph builder treats crystal water molecules as structural constraints. The algorithm will adapt the designed sequence to maintain favorable hydrogen-bonding networks with those specified waters.
* --select_with_and_with_without_ligands (Boolean): Instructs the algorithm to compute and output conditional probabilities for sequences both in the presence and in the absolute absence of the bound ligand, highlighting ligand-dependent design positions.


------------------------------
## 4. Sequence Conditioning & Side-Chain Masking
These parameters define how much of the native protein structure the algorithm is allowed to "see" or condition its predictions upon.

{
  "comment": "Conceptual algorithmic mask behavior inside run.py",
  "use_side_chain_embeddings": false,
  "conditional_prob_conditioning": "ordered_autoregressive"
}



* --pack_side_chains: Toggles whether the algorithm should run an internal post-processing step to predict packing configurations for the redesigned sequence residues around the target ligand.
* Chains to Design / Freeze: Algorithmically masks out specific segments of the structural graph. Unselected chains act as a fixed, unchanging electrostatic and steric background field that shapes the energy landscape of the designed regions.






In [ ]:
import atomworks
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

# Configure MPNN inference engine
# See mpnn.utils.inference.MPNN_GLOBAL_INFERENCE_DEFAULTS for all options
engine_config = {
    "model_type": "ligand_mpnn",  # or "protein_mpnn" for vanilla ProteinMPNN
    "is_legacy_weights": True,    # Required for now for ligand_mpnn and protein_mpnn
    "out_directory": f'{RESULTS_PATH}/{RESULTS_DIR}/ligand_mpnn',        # Return results in memory
    "write_structures": True,
    "write_fasta": True,
    "checkpoint_path": f"{FOUNDRY_PATH}/checkpoints/ligandmpnn_v_32_010_25.pt" # Explicitly set the path to the downloaded checkpoint
}

# Configure per-input inference options
# See mpnn.utils.inference.MPNN_PER_INPUT_INFERENCE_DEFAULTS for all options
input_configs = [
    {
        "batch_size": 20,         # Generate 20 sequences per structure
        "remove_waters": True,
        "seed": 3,
    }
]

# Load the CIF file into an AtomArray object
cif_file_path = f"{RESULTS_PATH}/{RESULTS_DIR}/{seleccionado}.cif.gz"
atom_array_obj = atomworks.io.parse(cif_file_path)
atom_array = atom_array_obj['assemblies']['1'][0]

# Run sequence design on the RFD3-generated backbone
model = MPNNInferenceEngine(**engine_config)
mpnn_outputs = model.run(input_dicts=input_configs, atom_arrays=[atom_array])

In [ ]:
from biotite.structure import get_residue_starts
from biotite.sequence import ProteinSequence

# Extract and display the designed sequences
print(f"Generated {len(mpnn_outputs)} designed sequences:\n")

for i, item in enumerate(mpnn_outputs):
    res_starts = get_residue_starts(item.atom_array)
    # Convert 3-letter codes to 1-letter using Biotite
    seq_1letter = ''.join(
        ProteinSequence.convert_letter_3to1(res_name)
        for res_name in item.atom_array.res_name[res_starts]
    )
    print(f"Sequence {i+1}: {seq_1letter}")

Generated 20 designed sequences:

Sequence 1: LHDQRIAVLEERTKSPDPRERARALIALAGEYIAKGEIEKAIAAAEEALASPDPELQIRGLGALGIAYAAKGDWERAEAALAEAEALAEGPMLGVALMARGEVLLMQGKTEEALAAFDRAAELLKDSPEYPEVLLARGRALLAQGRYEEALEDLDACLSLLPPDSFLHIEALLLRAEALEALGRTEEAARLRAEAAAALA
Sequence 2: LLDKEIAVLKKETKDPDPRKRAEALIKLAGVYIEQGKIDEAIEAAKKALASPDPALQVRGLGVLGIAYAAQGKWEEAEAALEEAEARATGAELGWALRARGIVLHMQGKHDEAFAAFERSLELNKDTPEYPEHLLAYGEALHDQGHYEEAAEYYRECLALLPPDSFLHLRALRRLAEVLEALGHHEEAARCRAEADAALA
Sequence 3: LLEKQKAELEAKTKSPDPRERARALIELARVYIEQGEIDEAIKAAEEALKSPDPELQIEGLGVLGVAYAKQGKWEEANAALDQALAAATGAMKGVALRYRAEVLHMQGKTEEALAAFRESLELLKDTPEYAEALLALGDALHDQGRHEEAIKYYEECLALLPPDSFLHIRALLSLADCLEALGRHEEAAECRARAAAALA
Sequence 4: LLDIAIKELEEKTKDPDPRKRAEALIELAEEYIAQGEIEKAIEAAKKALASPDPALQVRGLGVLGVAYAAQGKWDEANAALDAALAAATGAELGVALLYRGEVLLRQGKTEEALAAFRRSLELLRDTPEYPEALLALGEALLRQGRYEEAVEYFDRCLALLPPDSRLHIRALLARAEAYEALGRTEEAEKLRKEAEAAIK
Sequence 5: LLDLRIAELEAQTASPDPRERAEALIALAREYIKKGEIDKAVAAAEEALASPDPELQVRALGVLGIAYAAQGRWDEAEAALAEALAAATGATLGVALRARGR

In [ ]:
import torch

# Limpiar la cache de la GPU para liberar la memoria
torch.cuda.empty_cache()

In [ ]:
# Correr RF3 mediante la terminal linux
from Bio import SeqIO

In [ ]:
# Stream records one by one to save memory
INPUT_FASTA_PATH = f"{REPO_PATH}/outputs/UNK-lig"
INPUT_FASTA_DIR = f"{INPUT_FASTA_PATH}/ligand_mpnn"
#print(SeqIO.parse(f"{INPUT_FASTA_DIR}/unnamed.fa", "fasta"))
#for record in SeqIO.parse(f"{INPUT_FASTA_DIR}/unnamed.fa", "fasta"):
#    print(f"ID: {record.id}")
#    print(f"Description: {record.description}")
#    print(f"Sequence: {record.seq}\n")
# Abre el iterador y extrae únicamente el primer registro
record = next(SeqIO.parse(f"{INPUT_FASTA_DIR}/unnamed.fa", "fasta"))

# Muestra los datos en pantalla
print(f"ID: {record.id}")
print(f"Descripción: {record.description}")
print(f"Longitud: {len(record.seq)} pb")
print(f"Secuencia: {record.seq}")


ID: unnamed_b0_d0,
Descripción: unnamed_b0_d0, sequence_recovery=0.4485, ligand_interface_sequence_recovery=nan
Longitud: 136 pb
Secuencia: SSAALFAAGLEKYAEGDKKEAAGEVEEAIKAYEEAAKLFVASGTAAGLAYAEMAKAHIELLKGDVEGAKKHYEKAIELAKDDPITKGRALYELADLYLKEGKVEEAKKLYEEAVELVKDVDPAFAAEVEAKLKALS


In [ ]:
# Al correr esta celda la sesión se debe reiniciar y es necesario recargar
# las variables de entorno de los directorios y la conexión con Google Drive
!git clone https://github.com/google-deepmind/alphafold3.git
!pip install ./alphafold3/

Cloning into 'alphafold3'...
remote: Enumerating objects: 2499, done.
remote: Counting objects: 100% (982/982), done.
remote: Compressing objects: 100% (314/314), done.
remote: Total 2499 (delta 768), reused 668 (delta 668), pack-reused 1517 (from 2)
Receiving objects: 100% (2499/2499), 49.72 MiB | 28.85 MiB/s, done.
Resolving deltas: 100% (1531/1531), done.
Processing ./alphafold3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of flax to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.6/36.6 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.6/606.6 kB 

In [ ]:
import os, glob

USE_NATIVE = True
AF3_DIR = f'{HOME_PATH}/af3'
NATIVE_DIR = f'{AF3_DIR}/af3_native_weights'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'

#if not os.path.isfile(f'{AF3_DIR}/ALPHAFOLD3_READY'):
print('Installing packages...')
#os.system("pip install -q 'jax[cuda12]==0.10.1' dm-haiku==0.0.16 rdkit==2025.9.4 zstandard awscli tokamax==0.0.11 py3Dmol py2Dmol")
#os.system("pip install -q --no-deps 'https://github.com/sokrypton/alphafold3/releases/download/v3.1.4/alphafold3_open-3.1.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'")
base = 'https://raw.githubusercontent.com/sokrypton/alphafold3/refs/heads/main'
for script in ['run_alphafold', 'convert_of3_weights']:
  os.system(f'wget -q -nc {base}/{script}.py')
  os.system(f'touch {AF3_DIR}/ALPHAFOLD3_READY')
print('Packages installed.')

Installing packages...
Packages installed.


In [ ]:
# Patch tokamax so Ada/consumer GPUs (L4, A10, RTX 30/40; cc 8.6/8.9) fall back to XLA
# kernels. tokamax enables its Triton kernels for ALL cc>=8.0 GPUs, but those kernels
# need more shared memory than Ada cards have -> 'Shared memory size limit exceeded' at
# launch (which its trace-time fallback can't catch). Restrict Triton to true datacenter
# GPUs (A100 cc 8.0, H100 cc 9.0+); everything else uses XLA, exactly like the T4 path.
try:
  import tokamax
  _gu = os.path.join(os.path.dirname(tokamax.__file__), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 (8.6/8.9) lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights: official AlphaFold 3 (public download) or OpenFold3 (default, free) - both in background
if USE_NATIVE:
  if not os.path.isfile(f'{AF3_DIR}/NATIVE_WEIGHTS_DONE'):
    print('Downloading official AlphaFold 3 weights (public, no login required)...')
    os.makedirs(NATIVE_DIR, exist_ok=True)
    for _f in glob.glob(f'{NATIVE_DIR}/*'):       # keep exactly one model file in the dir
      os.remove(_f)
    os.system(f'(wget -q -O {NATIVE_DIR}/af3.bin.zst "{AF3_WEIGHTS_URL}"; touch {AF3_DIR}/NATIVE_WEIGHTS_DONE) &')
elif not os.path.isfile(f'{AF3_DIR}/WEIGHTS_DONE'):
  print('Downloading OpenFold3 weights...')
  os.system(f'(aws s3 cp s3://openfold/staging/of3-p2-155k.pt {AF3_DIR} --no-sign-request; touch {AF3_DIR}/WEIGHTS_DONE) &')

# Build AF3 data files (background, independent of weights)
if not os.path.isfile(f'{AF3_DIR}/DATA_DONE'):
  print('Building AF3 data files...')
  os.system(f'(build_data; touch {AF3_DIR}/DATA_DONE) &')

# Wait for background jobs
for sentinel in ([f'{AF3_DIR}/NATIVE_WEIGHTS_DONE', f'{AF3_DIR}/DATA_DONE'] if USE_NATIVE else [f'{AF3_DIR}/WEIGHTS_DONE', f'{AF3_DIR}/DATA_DONE']):
  while not os.path.isfile(sentinel):
    time.sleep(5)
  print(f'{sentinel} ✓')

if USE_NATIVE and os.path.getsize(f'{NATIVE_DIR}/af3.bin.zst') < 1_000_000:
  raise RuntimeError('AlphaFold 3 weights download failed or incomplete - re-run this cell.')

if not USE_NATIVE and not os.path.isfile('af3_converted_weights/of3_ported_weights.bin.zst'):
  print("converting openfold3 weights...")
  os.system("python convert_of3_weights.py --of3_checkpoint of3-p2-155k.pt --output_dir af3_converted_weights")

print('Setup complete!  ' + ('Using official AlphaFold 3 weights.' if USE_NATIVE else 'Using OpenFold3 weights.'))

/content/drive/MyDrive/bioinformatica/af3/NATIVE_WEIGHTS_DONE ✓
/content/drive/MyDrive/bioinformatica/af3/DATA_DONE ✓
Setup complete!  Using official AlphaFold 3 weights.


In [ ]:
!build_data after --no-deps

Parsing /usr/local/lib/python3.13/dist-packages/share/libcifpp/components.cif
100% 51056/51056 [00:10<00:00, 4645.97it/s]
Writing /usr/local/lib/python3.13/dist-packages/alphafold3/constants/converters/ccd.pickle
Done
Loading /usr/local/lib/python3.13/dist-packages/alphafold3/constants/converters/ccd.pickle
Finding ions and glycans
100% 51056/51056 [00:00<00:00, 194941.48it/s]
writing to /usr/local/lib/python3.13/dist-packages/alphafold3/constants/converters/chemical_component_sets.pickle
Done


In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: Tesla T4


In [ ]:
# ============================================================
# 1. DETECTAR DISPOSITIVO Y FLAGS
# ============================================================

import os
import subprocess


def detect_device():
    try:
        out = subprocess.run(
            ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=15)
        caps = [float(x) for x in out.stdout.split() if x.strip()]
        if caps:
            return 'gpu', min(caps)
    except Exception:
        pass
    return 'cpu', None

device, cap = detect_device()
nojit = False
xla_flags = []
if device == 'cpu':
    flash_impl = 'xla'
    nojit = True
    print('No GPU detected - running on CPU with XLA attention + --nojit (slow).')
elif cap < 8.0:
    flash_impl = 'xla'
    xla_flags = ['--xla_disable_hlo_passes=custom-kernel-fusion-rewriter']
    print(f'Pre-Ampere GPU (compute capability {cap}) - XLA attention + custom-kernel fusion disabled.')
elif 8.0 < cap < 9.0:
    flash_impl = 'xla'
    xla_flags = ['--xla_gpu_enable_triton_gemm=false']
    print(f'Ada/consumer GPU (cc {cap}) - XLA attention + Triton GEMM disabled.')
else:
    flash_impl = 'triton'
    xla_flags = ['--xla_gpu_enable_triton_gemm=false']
    print(f'Datacenter GPU (cc {cap}) - Triton flash attention + Triton GEMM disabled.')

cur = os.environ.get('XLA_FLAGS', '')
for f in xla_flags:
    if f not in cur:
        cur = (cur + ' ' + f).strip()
if cur:
    os.environ['XLA_FLAGS'] = cur


Pre-Ampere GPU (compute capability 7.5) - XLA attention + custom-kernel fusion disabled.


In [ ]:
#@title 🚀 Procesamiento por lotes de secuencias FASTA con AlphaFold3 (soporte para complejos y enlaces covalentes)
import re
import json
import hashlib
import glob
import shutil
import pandas as pd
import zipfile
from google.colab import files
from IPython.display import display, HTML
import ipywidgets as widgets
import time


In [ ]:
# ============================================================
# PARÁMETROS GLOBALES
# ============================================================

ligand_ccd = ""

ligand_smiles = "C(=N\[C@H](C(=O)N[C@H](C(=O)N[C@H](C(=O)N[C@H](C(=O)N1[C@H](C(=O)N[C@H](C(=O)O)CCC(=O)O)CCC1)[C@@H](C)O)CC(C)C)Cc1cnc[nH]1)C(C)C)/[C@]1([C@H]([C@@H]([C@H](O1)CO)O)O)O"

msa_mode = "single_sequence"
seeds = "1"
num_recycles = 2
num_diffusion_samples = 3
on_existing = "overwrite"

batch_name = f"{mejores[8]}_AF3"

print(batch_name)


backbone_0_3_model_1_AF3


<>:7: SyntaxWarning: invalid escape sequence '\['
<>:7: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_17563/709585011.py:7: SyntaxWarning: invalid escape sequence '\['
  ligand_smiles = "C(=N\[C@H](C(=O)N[C@H](C(=O)N[C@H](C(=O)N[C@H](C(=O)N1[C@H](C(=O)N[C@H](C(=O)O)CCC(=O)O)CCC1)[C@@H](C)O)CC(C)C)Cc1cnc[nH]1)C(C)C)/[C@]1([C@H]([C@@H]([C@H](O1)CO)O)O)O"


In [ ]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================
def split_entries(s):
    s = re.sub(r':+', ':', s).strip(':') # limpia la cadena unificando grupos de dos o más dos puntos (:) consecutivos en uno solo y eliminando dos puntos al inicio o al final del texto
    return [e for e in (''.join(tok.split()) for tok in s.split(':')) if e] #

def get_seed_list(seeds_str):
    seed_list = []
    for tok in re.findall(r'\d+', seeds_str):
        v = int(tok)
        if v not in seed_list:
            seed_list.append(v)
    if not seed_list:
        seed_list = [1]
    return seed_list

def build_json_for_sequence(seq_str, ligand_ccd='', ligand_smiles='', msa_mode='mmseqs2_server', seed_list=[1], covalent_bond=''):
    """Construye el JSON de entrada para AF3, manejando múltiples cadenas y enlaces covalentes."""
    # Limpiar y dividir por ':'
    seq_str = re.sub(r'\s+', '', seq_str)
    chain_seqs = [s for s in seq_str.split(':') if s]
    if not chain_seqs:
        raise ValueError("La secuencia está vacía.")

    chains = []
    idx = 0
    # Asignar IDs A, B, C, ...
    for chain_seq in chain_seqs:
        cid = chr(ord('A') + idx)
        idx += 1
        prot_seq = chain_seq.upper()
        # Validar que solo contiene letras A-Z
        if not re.match(r'^[A-Z]+$', prot_seq):
            raise ValueError(f"Secuencia de proteína inválida: {prot_seq}")
        ent = {'id': cid, 'sequence': prot_seq, 'templates': []}
        if msa_mode == 'single_sequence':
            ent.update({'unpairedMsa': f'>query\n{prot_seq}\n', 'pairedMsa': ''})
        chains.append({'protein': ent})
        print(f"   Cadena {cid}: {prot_seq[:30]}... ({len(prot_seq)} aa)")

    # Añadir ligandos (si se especifican) con IDs subsiguientes
    ligand_id = None
    if ligand_ccd:
        ligand_id = chr(ord('A') + idx)
        chains.append({'ligand': {'id': ligand_id, 'ccdCodes': [ligand_ccd]}})
        print(f"   Ligando CCD: {ligand_ccd} (ID {ligand_id})")
        idx += 1
    if ligand_smiles:
        ligand_id = chr(ord('A') + idx)
        chains.append({'ligand': {'id': ligand_id, 'smiles': ligand_smiles}})
        print(f"   Ligando SMILES: {ligand_smiles[:30]}... (ID {ligand_id})")
        idx += 1

    fold_input = {
        'name': 'job',  # se reemplazará
        'sequences': chains,
        'modelSeeds': seed_list,
        'dialect': 'alphafold3',
        'version': 1,
    }

    # === Añadir enlace covalente si se especifica ===
    if covalent_bond and ligand_id:
        parts = covalent_bond.split(':')
        if len(parts) == 6:
            lig_chain, lig_res, lig_atom, prot_chain, prot_res, prot_atom = parts
            try:
                lig_res = int(lig_res)
                prot_res = int(prot_res)
            except ValueError:
                print(f"   ⚠️ Los números de residuo deben ser enteros: {covalent_bond}. Se ignorará.")
            else:
                fold_input['bondedAtomPairs'] = [
                    [
                        [lig_chain, lig_res, lig_atom],
                        [prot_chain, prot_res, prot_atom]
                    ]
                ]
                print(f"   🔗 Enlace covalente: {lig_chain}:{lig_res}@{lig_atom} - {prot_chain}:{prot_res}@{prot_atom}")
        else:
            print(f"   ⚠️ Formato de enlace inválido: {covalent_bond}. Se ignorará.")
    elif covalent_bond and not ligand_id:
        print("   ⚠️ No se puede añadir enlace covalente porque no hay ligando definido.")

    return fold_input


In [ ]:
# ============================================================
# 2. PARSEAR TODAS LAS SECUENCIAS DEL ARCHIVO FASTA
# ============================================================
INPUT_FASTA_DIR = f"{REPO_PATH}/outputs/UNK-lig/ligand_mpnn"
fasta_files = [f"{INPUT_FASTA_DIR}/unnamed.fa"]
sequences = []                                                                      # lista de (nombre, secuencia)
for fpath in fasta_files:                                                           # Toma el nombre del archivo de la lista "fasta_files"
    with open(fpath, 'r') as f:                                                     # Abre el archivo
        lines = f.readlines()                                                       # Copia las lineas en la variable "lines"
    current_seq = ""                                                                # Crea la variable current_seq
    current_name = ""                                                               # Crea la variable current_name
    for line in lines:                                                              # Para cada linea de "lines"
        line = line.strip()                                                         # Limpia el string de caracteres repetidos
        if line.startswith('>'):                                                    # Si la linea empieza con ">"
            if current_seq and current_name:                                        # Si "current_seq" y "current_name" no están vacias
                sequences.append((current_name, current_seq))                       # agrega los valores a sequences (lista) como tupla
            current_name = line[1:].split()[0]                                      # Si no asigna a current_name el valor de la linea empezando por el segundo valor y tomando solo el primer elemento
            current_seq = ""                                                        # "current_seq" queda vacio porque no es la linea que tiene la secuencia
        else:                                                                       # si la linea no empieza con ">" entonces:
            current_seq += line                                                     # A la variable "current_seq" se le asigna la linea porque tiene la secuencia
    if current_seq and current_name:                                                # Si current_seq y current_name no estan vacias se le asigna sus valores a sequence en forma de tupla
        sequences.append((current_name, current_seq))

print(f"\n📄 Total de secuencias encontradas: {len(sequences)}")                    # Imprime en pantalla el numero de secuencias encontradas
for name, seq in sequences[:5]:                                                     # Muestra las primeras 5 secuencias
    display_seq = seq[:30] + '...' if len(seq) > 30 else seq                        # Muestra los 30 primeros aa si la secuencia es > a 30 seguido de ...
    num_chains = len(seq.split(':')) if ':' in seq else 1                           # Muestra el nro de cadenas en la secuencia
    print(f"   - {name}: {display_seq} ({len(seq)} aa, {num_chains} cadena(s))")    # Imprime el nombre de la secuencia, el tamaño y el numero de cadenas
if len(sequences) > 5:                                                              # Si las secuencias enconntradas son mayores que 5 entonces nos indica cuantas mas hay
    print(f"   ... y {len(sequences)-5} más.")


📄 Total de secuencias encontradas: 20
   - unnamed_b0_d0,: LHDQRIAVLEERTKSPDPRERARALIALAG... (200 aa, 1 cadena(s))
   - unnamed_b0_d1,: LLDKEIAVLKKETKDPDPRKRAEALIKLAG... (200 aa, 1 cadena(s))
   - unnamed_b0_d2,: LLEKQKAELEAKTKSPDPRERARALIELAR... (200 aa, 1 cadena(s))
   - unnamed_b0_d3,: LLDIAIKELEEKTKDPDPRKRAEALIELAE... (200 aa, 1 cadena(s))
   - unnamed_b0_d4,: LLDLRIAELEAQTASPDPRERAEALIALAR... (200 aa, 1 cadena(s))
   ... y 15 más.


In [ ]:
# ============================================================
# 3. CONFIGURAR PARÁMETROS DE EJECUCIÓN
# ============================================================
#seed_list = get_seed_list(seeds)                                                      # Crea una lista de la semilla y la guarda en seed_list
seed_list = [1,2,3,4]
basejob = re.sub(r'\W+', '', batch_name) or "batch"                                   # Elimina los caracteres que no sean alfanumericos o "_" y los guarda en basejob
batch_jobname = basejob + "_" + hashlib.sha1(str(seed_list).encode()).hexdigest()[:5] # Agrega a basejob un hash de 5 caracteres generado a partir de la semilla
OUTPUT_DIR = f'{REPO_PATH}/af3_output'                                                # Carpeta de salida
BATCH_DIR = f"{OUTPUT_DIR}/{batch_jobname}"                                           # Carpeta del trabajo
os.makedirs(BATCH_DIR, exist_ok=True)                                                 # Crea la carpeta

In [ ]:
# ============================================================
# 4. EJECUTAR AF3 PARA CADA SECUENCIA
# ============================================================
use_of3 = not (os.path.isdir(f'{AF3_DIR}/af3_native_weights') and glob.glob(f'{AF3_DIR}/af3_native_weights/*.bin*'))
model_dir = f'{AF3_DIR}/af3_native_weights' if not use_of3 else f'{AF3_DIR}/af3_converted_weights'

metrics_list = []

for idx, (name, seq) in enumerate(sequences):
    print(f"\n🔹 Procesando {idx+1}/{len(sequences)}: {name}")

    try:
        # Construir JSON (ahora con soporte para múltiples cadenas y enlace covalente)
        fold_input = build_json_for_sequence(
            seq,
            #ligand_ccd=ligand_ccd,
            ligand_smiles=ligand_smiles,
            msa_mode=msa_mode,
            seed_list=seed_list,
            #covalent_bond=covalent_bond
            covalent_bond=''
        )
    except ValueError as e:
        print(f"   ❌ Error en la secuencia: {e}")
        metrics_list.append({'jobname': name, 'name': name, 'error': str(e)})
        continue

    # Generar jobname único
    jobname = f"{batch_jobname}_{idx:03d}_{re.sub(r'\W+', '', name)[:20]}"
    fold_input['name'] = jobname

    # Guardar JSON
    INPUT_DIR = '/tmp/af3_inputs'
    os.makedirs(INPUT_DIR, exist_ok=True)
    json_path = f'{INPUT_DIR}/{jobname}.json'
    with open(json_path, 'w') as f:
        json.dump(fold_input, f, indent=2)

    # Definir carpeta de salida
    job_dir = f'{BATCH_DIR}/{jobname}'

    # Verificar si ya existe (on_existing)
    if on_existing == 'skip' and os.path.isdir(job_dir) and any(f.endswith('.cif') for f in os.listdir(job_dir)):
        print(f"   ⏩ Saltando (ya existe): {job_dir}")
        conf_path = f'{job_dir}/{jobname}_confidences.json'
        if os.path.exists(conf_path):
            with open(conf_path, 'r') as f:
                conf = json.load(f)
            summ_path = f'{job_dir}/{jobname}_summary_confidences.json'
            summ = {}
            if os.path.exists(summ_path):
                with open(summ_path, 'r') as f:
                    summ = json.load(f)
            metrics_list.append({
                'jobname': jobname,
                'name': name,
                'mean_plddt': summ.get('mean_plddt', None),
                'ptm': summ.get('ptm', None),
                'iptm': summ.get('iptm', None),
                'ranking_score': summ.get('ranking_score', None),
            })
        continue

    # Limpiar carpeta si overwrite
    if on_existing == 'overwrite':
        shutil.rmtree(job_dir, ignore_errors=True)

    # Comando
    cmd = [
        'python', 'run_alphafold.py',
        f'--json_path={json_path}',
        f'--model_dir={model_dir}',
        '--norun_data_pipeline',
        f'--output_dir={BATCH_DIR}',
        '--cache_dir=/tmp/af3_cache',
        '--force_output_dir',
        f'--flash_attention_implementation={flash_impl}',
        f'--num_recycles={num_recycles}',
        f'--num_diffusion_samples={num_diffusion_samples}',
    ]
    if msa_mode == 'mmseqs2_server':
        cmd.append('--use_msa_server')
    if nojit:
        cmd.append('--nojit')
    if use_of3:
        cmd.append('--of3_weights')

    cmd_str = ' '.join(cmd)
    print(f"   Comando: {cmd_str}")
    print(f"   Salida: {job_dir}/")

    # Ejecutar
    start = time.time()
    result = subprocess.run(cmd_str, shell=True, capture_output=True, text=True)
    elapsed = time.time() - start

    if result.returncode != 0:
        print(f"   ❌ Error: {result.stderr}")
        metrics_list.append({
            'jobname': jobname,
            'name': name,
            'mean_plddt': None,
            'ptm': None,
            'iptm': None,
            'ranking_score': None,
            'error': result.stderr[:300] + '...' if len(result.stderr) > 300 else result.stderr
        })
        continue

    print(f"   ✅ Completado en {elapsed:.1f}s")

    # Extraer métricas
    conf_path = f'{job_dir}/{jobname}_confidences.json'
    summ_path = f'{job_dir}/{jobname}_summary_confidences.json'
    if os.path.exists(conf_path):
        with open(conf_path, 'r') as f:
            conf = json.load(f)
        summ = {}
        if os.path.exists(summ_path):
            with open(summ_path, 'r') as f:
                summ = json.load(f)
        metrics_list.append({
            'jobname': jobname,
            'name': name,
            'mean_plddt': summ.get('mean_plddt', None),
            'ptm': summ.get('ptm', None),
            'iptm': summ.get('iptm', None),
            'ranking_score': summ.get('ranking_score', None),
        })
    else:
        print(f"   ⚠️ No se encontró archivo de confianza para {jobname}")
        metrics_list.append({
            'jobname': jobname,
            'name': name,
            'mean_plddt': None,
            'ptm': None,
            'iptm': None,
            'ranking_score': None,
        })



🔹 Procesando 1/20: unnamed_b0_d0,
   Cadena A: LHDQRIAVLEERTKSPDPRERARALIALAG... (200 aa)
   Ligando SMILES: C(=N\[C@H](C(=O)N[C@H](C(=O)N[... (ID B)
   Comando: python run_alphafold.py --json_path=/tmp/af3_inputs/backbone_0_3_model_1_AF3_d745e_000_unnamed_b0_d0.json --model_dir=/content/drive/MyDrive/bioinformatica/af3/af3_native_weights --norun_data_pipeline --output_dir=/content/drive/MyDrive/bioinformatica/foundry/gas04/af3_output/backbone_0_3_model_1_AF3_d745e --cache_dir=/tmp/af3_cache --force_output_dir --flash_attention_implementation=xla --num_recycles=2 --num_diffusion_samples=3
   Salida: /content/drive/MyDrive/bioinformatica/foundry/gas04/af3_output/backbone_0_3_model_1_AF3_d745e/backbone_0_3_model_1_AF3_d745e_000_unnamed_b0_d0/
   ✅ Completado en 657.7s

🔹 Procesando 2/20: unnamed_b0_d1,
   Cadena A: LLDKEIAVLKKETKDPDPRKRAEALIKLAG... (200 aa)
   Ligando SMILES: C(=N\[C@H](C(=O)N[C@H](C(=O)N[... (ID B)
   Comando: python run_alphafold.py --json_path=/tmp/af3_inputs/backbon

In [ ]:
# ============================================================
# 5. RESUMEN Y TABLA DE MÉTRICAS
# ============================================================
if metrics_list:
    df = pd.DataFrame(metrics_list)
    if 'ranking_score' in df.columns and df['ranking_score'].notna().any():
        df = df.sort_values(by='ranking_score', ascending=False)
    else:
        df = df.sort_values(by='mean_plddt', ascending=False, na_position='last')
    print("\n📊 Tabla de métricas para todas las predicciones:")
    display(df)

    csv_path = os.path.join(BATCH_DIR, "metricas_todas.csv")
    df.to_csv(csv_path, index=False)
    print(f"✅ CSV guardado en: {csv_path}")
else:
    print("⚠️ No se generaron métricas.")

print("\n✅ Procesamiento completado.")



NameError: name 'metrics_list' is not defined

In [ ]:
from atomworks.io.parser import parse
# Shared utilities for visualization (from AtomWorks)
from atomworks.io.utils.visualize import view


#OUTPUT_DIR = f'{REPO_PATH}/af3_output'                            # Carpeta de salida
#BATCH_DIR = f"{OUTPUT_DIR}/{batch_jobname}"                       # Carpeta del trabajo

DIR_NAME=f"backbone_0_8_model_7_AF3_f629a_004_unnamed_b0_d4"

resultAF3 = parse(f"{BATCH_DIR}/{DIR_NAME}/{DIR_NAME}_model.cif")
resultAF3.keys()
#result
atom_array = resultAF3['assemblies']['1'][0]
#print(atom_array)
view(atom_array)

ModuleNotFoundError: No module named 'atomworks'

In [ ]:
# Load CIF data from a local file

FILE = "/content/drive/MyDrive/bioinformatica/foundry/gas01/af3_output/backbone_0_8_model_6_AF3_f629a/backbone_0_8_model_6_AF3_f629a_002_unnamed_b0_d2/backbone_0_8_model_6_AF3_f629a_002_unnamed_b0_d2_model.cif"
with open(f"{FILE}", "rt") as f:
    cif_data = f.read()

# Initialize viewer
view = py3Dmol.view(width=800, height=600)

# Add CIF model
view.addModel(cif_data, "cif")

# Set style (e.g., stick representation)
view.setBackgroundColor('black')
view.setStyle({'chain':'A'}, {'cartoon': {'color': 'spectrum'}})
view.setStyle({'chain':'B'}, {'stick': {'colorscheme':'yellowCarbon'}})
view.addSurface(py3Dmol.VDW, {'opacity':0.6, 'color':'grey'}, \
  {'not':{'chain':'B'}})
#view.setStyle({"stick": {}})

# Zoom and render
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.